<a href="https://colab.research.google.com/github/gcalanch/DMA-Caras/blob/main/PreProcesamiento_GCv2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits  # Ejemplo, reemplázalo con tus datos
from sklearn.manifold import Isomap
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, make_scorer

# --------------------------------------
# 1. Cargar datos (reemplazar esto con datos)
# --------------------------------------

In [ ]:
data = load_digits()  # Dataset de ejemplo con etiquetas
X = data.data  # (1797 muestras, 64 características)
y = data.target

# Si tienes tus propios datos, reemplaza X, y aquí:
# X = tus_datos
# y = tus_etiquetas

In [10]:
# prompt: como indicar la busqueda si los datos estan en el drive https://drive.google.com/drive/folders/1Lpwwsm-1dm0qgyFEZ9xZYZQZBjOpFGcA. Dentro de esa direccion del drive los datos son archivos que estan ubicados en carpetas. Estas carpetas deben ser las etiquetas de los datos que se deben agegar. corregir el Error al leer el archivo /content/drive/MyDrive/DMA/Caras/Noelia R/IMG_1545.JPG: 'utf-8' codec can't decode byte 0xff in position 0: invalid start byte

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits  # Ejemplo, reemplázalo con tus datos
from sklearn.manifold import Isomap
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, make_scorer
import os
from PIL import Image
import pandas as pd

# Monta tu Google Drive
from google.colab import drive
drive.mount('/content/drive')

# --------------------------------------
# 1. Cargar datos desde Google Drive
# --------------------------------------

# Ruta a la carpeta principal en tu Google Drive
root_folder = "/content/drive/MyDrive/DMA/Caras/"

# Inicializa listas para almacenar las imágenes y las etiquetas
images = []
labels = []


for folder_name in os.listdir(root_folder):
  folder_path = os.path.join(root_folder, folder_name)
  if os.path.isdir(folder_path):
    label = folder_name
    for filename in os.listdir(folder_path):
      filepath = os.path.join(folder_path, filename)
      try:
          # Abre la imagen con Pillow y conviértela a escala de grises
          img = Image.open(filepath).convert('L')  # 'L' para escala de grises
          img_array = np.array(img).flatten() # Aplana la imagen a un vector
          images.append(img_array)
          labels.append(label)
      except Exception as e:
          print(f"Error al leer el archivo {filepath}: {e}")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Error al leer el archivo /content/drive/MyDrive/DMA/Caras/Cristian/._Foto4.png: cannot identify image file '/content/drive/MyDrive/DMA/Caras/Cristian/._Foto4.png'
Error al leer el archivo /content/drive/MyDrive/DMA/Caras/Cristian/._Foto2.png: cannot identify image file '/content/drive/MyDrive/DMA/Caras/Cristian/._Foto2.png'
Error al leer el archivo /content/drive/MyDrive/DMA/Caras/Cristian/._Foto6.png: cannot identify image file '/content/drive/MyDrive/DMA/Caras/Cristian/._Foto6.png'
Error al leer el archivo /content/drive/MyDrive/DMA/Caras/Cristian/._Foto5.png: cannot identify image file '/content/drive/MyDrive/DMA/Caras/Cristian/._Foto5.png'
Error al leer el archivo /content/drive/MyDrive/DMA/Caras/Cristian/._Foto7.png: cannot identify image file '/content/drive/MyDrive/DMA/Caras/Cristian/._Foto7.png'
Error al leer el archivo /content/drive/MyDrive/DMA/Cara

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (740,) + inhomogeneous part.

In [11]:
# prompt: quisiera generar codigo para la lectura de datos desde un archivo datos_isomap.pkl que esta en la carpeta DMA del drive

import pickle

# Ruta al archivo .pkl en tu Google Drive
file_path = "/content/drive/MyDrive/DMA/datos_isomap.pkl"

try:
  with open(file_path, 'rb') as file:
    data = pickle.load(file)
    # 'data' ahora contiene los datos cargados desde el archivo .pkl
    print("Datos cargados correctamente desde:", file_path)
    # Puedes acceder a los datos cargados, por ejemplo:
    # if isinstance(data, dict):
    #   X = data['X']
    #   y = data['y']
    #   print(X.shape)
    #   print(y.shape)
    # elif isinstance(data, tuple):
    #   X, y = data
    #   print(X.shape)
    #   print(y.shape)
    # else:
    #   print("Formato de datos desconocido")
except FileNotFoundError:
  print(f"Error: Archivo no encontrado en {file_path}")
except Exception as e:
  print(f"Error al cargar los datos: {e}")


Datos cargados correctamente desde: /content/drive/MyDrive/DMA/datos_isomap.pkl


In [12]:

# Convierte las listas a arrays numpy
X = np.array(images)
y = np.array(labels)

# Verifica la forma de los datos y las etiquetas
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)
# Resto del código...


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (740,) + inhomogeneous part.

# --------------------------------------
# 2. Definir el pipeline: ISOMAP + KNN
# --------------------------------------

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),           # Normalización (opcional, recomendable)
    ('isomap', Isomap()),
    ('knn', KNeighborsClassifier())
])

# --------------------------------------
# 3. Definir parámetros para GridSearch
# --------------------------------------

In [ ]:
param_grid = {
    'isomap__n_neighbors': [5, 10, 15],
    'isomap__n_components': [2, 3, 5, 10],
    'knn__n_neighbors': [3, 5, 7]
}

# --------------------------------------
# 4. Búsqueda con GridSearch y Cross-Validation
# --------------------------------------

In [ ]:
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
scorer = make_scorer(accuracy_score)

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring=scorer,
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X, y)

print("\n✅ Mejor accuracy promedio (50 evaluaciones): {:.4f}".format(grid_search.best_score_))
print("🏆 Mejores parámetros encontrados:", grid_search.best_params_)

# --------------------------------------
# 5. Visualizar el mejor embedding en 2D
# --------------------------------------

In [ ]:
# Entrenamos el mejor modelo de ISOMAP para visualizar el embedding
best_isomap = Isomap(
    n_neighbors=grid_search.best_params_['isomap__n_neighbors'],
    n_components=2  # Visualización 2D
)

X_scaled = StandardScaler().fit_transform(X)
X_embedded = best_isomap.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_embedded[:, 0], X_embedded[:, 1], c=y, cmap='Spectral', s=20)
plt.title("🔍 Mejor Embedding ISOMAP (2D)")
plt.xlabel("Componente 1")
plt.ylabel("Componente 2")
plt.colorbar(scatter, label='Etiqueta')
plt.grid(True)
plt.tight_layout()
plt.show()